Libraries

In [107]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import time
from sklearn.preprocessing import OneHotEncoder

In [108]:
mnist_df = fetch_openml('mnist_784', version= 1, as_frame= False)

X = mnist_df.data.astype(np.float32) / 255.0
y = mnist_df.target.astype(np.int64)

In [109]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=22,stratify=y)

In [110]:
X_test

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [111]:
y_train

array([4, 1, 7, ..., 1, 0, 7])

In [112]:
import pandas as pd

df = pd.DataFrame(X)
df['label'] = y

print(df.head())

#labels are natural numbers from 0 to 9 i will one hot encode them so we can use it for loss function

     0    1    2    3    4    5    6    7    8    9  ...  775  776  777  778  \
0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
4  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   

   779  780  781  782  783  label  
0  0.0  0.0  0.0  0.0  0.0      5  
1  0.0  0.0  0.0  0.0  0.0      0  
2  0.0  0.0  0.0  0.0  0.0      4  
3  0.0  0.0  0.0  0.0  0.0      1  
4  0.0  0.0  0.0  0.0  0.0      9  

[5 rows x 785 columns]


In [113]:
encoder = OneHotEncoder(sparse_output= False, categories='auto')
y_train = encoder.fit_transform(y_train.reshape(-1,1))
y_test = encoder.transform(y_test.reshape(-1,1))

y_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]])

In [114]:
def ReLU(x):
    return np.maximum(0,x)

In [115]:
h = 1e-8

def ord_derivative(function,x):
    return (function(h+x) - function(x))/h

In [ ]:
def softmax_function(x):
    x_shift = x - np.max(x, axis=1, keepdims=True)
    expx = np.exp(x_shift)
    return expx / np.sum(expx, axis=1, keepdims=True)

In [ ]:
def cross_entropy_loss(Ypred, Y):
    return -np.mean(np.sum(Y * np.log(Ypred + h), axis=1))
#we are adding h to prediction in case we get a zero for it. ln0 goes to negative infinity

In [ ]:
def accuracy(Ypred, Y):
    return np.mean(np.argmax(Ypred, axis=1) == np.argmax(Y, axis=1))


In [119]:


# ---------------------- 3) Neural network training ----------------------
def train_simple_nn(
    X_train, y_train, X_test, y_test,
    hidden_sizes=(128, 64),
    epochs=5, batch_size=128, lr=0.1, train_subset=None
):
    n_train, input_dim = X_train.shape
    num_classes = len(np.unique(y_train))
    
    if train_subset is not None and train_subset < n_train:
        idx = np.random.permutation(n_train)[:train_subset]
        X_train = X_train[idx]
        y_train = y_train[idx]
        n_train = X_train.shape[0]
    
    Y_train = one_hot(y_train, num_classes)
    Y_test = one_hot(y_test, num_classes)

    # Initialize weights (He init for ReLU)
    H1, H2 = hidden_sizes
    W1 = np.random.randn(input_dim, H1) * np.sqrt(2.0 / input_dim)
    b1 = np.zeros((1, H1), dtype=np.float32)
    W2 = np.random.randn(H1, H2) * np.sqrt(2.0 / H1)
    b2 = np.zeros((1, H2), dtype=np.float32)
    W3 = np.random.randn(H2, num_classes) * np.sqrt(2.0 / H2)
    b3 = np.zeros((1, num_classes), dtype=np.float32)

    loss_history = []
    test_acc_history = []
    start_time = time.time()

    for epoch in range(1, epochs+1):
        perm = np.random.permutation(n_train)
        X_shuf = X_train[perm]
        Y_shuf = Y_train[perm]
        epoch_loss = 0.0

        for i in range(0, n_train, batch_size):
            Xb = X_shuf[i:i+batch_size]
            Yb = Y_shuf[i:i+batch_size]
            m = Xb.shape[0]

            # Forward pass
            Z1 = Xb.dot(W1) + b1
            A1 = relu(Z1)
            Z2 = A1.dot(W2) + b2
            A2 = relu(Z2)
            Z3 = A2.dot(W3) + b3
            A3 = softmax(Z3)

            # Loss
            loss = cross_entropy_loss(A3, Yb)
            epoch_loss += loss * m

            # Backward pass (softmax + cross-entropy simplifies)
            dZ3 = (A3 - Yb) / m
            dW3 = A2.T.dot(dZ3)
            db3 = np.sum(dZ3, axis=0, keepdims=True)

            dA2 = dZ3.dot(W3.T)
            dZ2 = dA2 * relu_deriv(Z2)
            dW2 = A1.T.dot(dZ2)
            db2 = np.sum(dZ2, axis=0, keepdims=True)

            dA1 = dZ2.dot(W2.T)
            dZ1 = dA1 * relu_deriv(Z1)
            dW1 = Xb.T.dot(dZ1)
            db1 = np.sum(dZ1, axis=0, keepdims=True)

            # Update parameters
            W3 -= lr * dW3; b3 -= lr * db3
            W2 -= lr * dW2; b2 -= lr * db2
            W1 -= lr * dW1; b1 -= lr * db1

        epoch_loss /= n_train
        # Evaluate on test
        Z1t = X_test.dot(W1) + b1
        A1t = relu(Z1t)
        Z2t = A1t.dot(W2) + b2
        A2t = relu(Z2t)
        Z3t = A2t.dot(W3) + b3
        A3t = softmax(Z3t)
        test_acc = accuracy(A3t, Y_test)
        loss_history.append(epoch_loss)
        test_acc_history.append(test_acc)

        print(f"Epoch {epoch}/{epochs} - Loss: {epoch_loss:.4f} - Test Acc: {test_acc*100:.2f}%")

    total_time = time.time() - start_time
    print(f"Training finished in {total_time:.1f}s")
    return (W1,b1,W2,b2,W3,b3), loss_history, test_acc_history

# ---------------------- 4) Run training ----------------------
if __name__ == "__main__":
    params, loss_hist, acc_hist = train_simple_nn(
        X_train, y_train, X_test, y_test,
        hidden_sizes=(128, 64),
        epochs=5,
        batch_size=128,
        lr=0.12,
        train_subset=20000   # optional: limit training for speed
    )


NameError: name 'one_hot' is not defined